## Now import ModelFlow

In [1]:
import sys
if sys.platform == 'emscripten':  # only in the browser (JupyterLite): install ModelFlow, see mfsetup.py
    %run mfsetup.py
    await install_modelflow()

In [2]:
from modelclass import model

## Load the Pakistan model and run the baseline

In [3]:
mpak,bline = model.modelload('data/pak.pcim',run=True,keep='Baseline')

Zipped file read:  data\pak.pcim


In [4]:
mpak.PAKCCEMISCO2CKN

Endogeneous: PAKCCEMISCO2CKN: Coal emissions, tCO2e
Formular: FRML <IDENT> PAKCCEMISCO2CKN = (1000*EMISCOAL)*(PAKNVCOLPRODQN+PAKNVCOLNIMPQN) $

PAKCCEMISCO2CKN: Coal emissions, tCO2e
EMISCOAL       : 
PAKNVCOLNIMPQN : Coal, net import (ktoe)
PAKNVCOLPRODQN : Coal, production (ktoe)

Values :


,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
Base,"25,012,089.24","42,705,000.56","44,509,676.94","47,524,592.83","49,325,933.95","50,582,181.36","51,614,522.33","52,720,732.13","53,984,831.67","55,377,238.19","56,828,957.28","58,282,714.30","59,710,259.82","61,108,811.31","62,489,130.02"
Last,"25,012,089.24","42,705,000.56","44,509,676.94","47,524,592.83","49,325,933.95","50,582,181.36","51,614,522.33","52,720,732.13","53,984,831.67","55,377,238.19","56,828,957.28","58,282,714.30","59,710,259.82","61,108,811.31","62,489,130.02"
Diff,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Input last run:


,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030
EMISCOAL,3.96,3.96,3.96,3.96,3.96,3.96,3.96,3.96,3.96,3.96,3.96,3.96,3.96,3.96,3.96
PAKNVCOLNIMPQN,"4,624.17","9,062.63","9,531.70","10,262.67","10,710.28","11,021.36","11,267.81","11,518.35","11,793.88","12,089.70","12,391.70","12,687.95","12,972.76","13,246.09","13,510.99"
PAKNVCOLPRODQN,"1,690.88","1,719.52","1,706.10","1,736.33","1,743.52","1,749.62","1,763.81","1,792.57","1,836.20","1,891.93","1,956.46","2,027.26","2,102.88","2,182.65","2,266.26"


## Simulating the impact of imposing a carbon price

A simulation follows three steps:

1. **Create a scenario DataFrame** from a baseline using `.upd()`, which applies an update expression that sets, scales, or grows selected variables over a chosen time range.
2. **Solve the model** by calling it with the scenario DataFrame, a start and end year, and a `keep=` label that names the scenario for later comparison.
3. **Inspect results** on the returned DataFrame or directly on the model, which stores each kept scenario.

In [5]:
CT30df = bline.upd("<2025 2050> PAKGGREVCO2CER PAKGGREVCO2GER PAKGGREVCO2OER = 30")

**Breakdown** 


$$
\begin{aligned}
&\underbrace{\texttt{CT30df}}_{\text{scenario df}}
 = \underbrace{\texttt{bline}}_{\text{baseline}}
 \,.\,\underbrace{\texttt{upd}}_{\text{accessor}}\bigl(\texttt{"}\ldots\texttt{"}\bigr) \\[8pt]
&\text{where the update string is} \\[4pt]
&\texttt{"}\,
 \overbrace{\texttt{<2025 2050>}}^{\text{years}}\ 
 \overbrace{\texttt{PAKGGREVCO2CER PAKGGREVCO2GER PAKGGREVCO2OER}}^{\text{coal, gas, oil CO}_2\text{ tax rates}}\ 
 \overbrace{\texttt{=}}^{\text{set}}\ 
 \overbrace{\texttt{30}}^{\text{USD/tCO}_2}
 \,\texttt{"}
\end{aligned}
$$

### Solve the model

In [6]:
resultsdf = mpak(CT30df,2020,2050,keep="Nominal $30USD Carbon tax")

**Breakdown**
$$
\small
\underbrace{\texttt{resultsdf}}_{\text{results df}}
= \underbrace{\texttt{mpak}}_{\text{model}}\bigl(
\underbrace{\texttt{CT30df}}_{\text{scenario df}},\;
\underbrace{\texttt{2020}}_{\text{start}},\;
\underbrace{\texttt{2050}}_{\text{end}},\;
\underbrace{\texttt{keep="Nominal \$30USD Carbon tax"}}_{\text{scenario label}}
\bigr)
$$

### Readable labels for the emission variables

To make tables and plots easier to read, we attach short descriptions to each emission variable and merge them into `mpak.var_description`. These labels can then be used instead of the raw codes wherever the model displays results.

### Inspect results

 **Note**
 
 `resultsdf` is rarely used directly. The model object `mpak` keeps the baseline (`mpak.basedf`) and the latest run (`mpak.lastdf`) internally, so the following cells can compare and plot results directly from `mpak`.

In [7]:
mpak['PAKNYGDPMKTPCN PAKFMLBLPOLYXN PAKCCEMISCO2?KN']

In [8]:
mpak.PAKNYGDPMKTPCN

Endogeneous: PAKNYGDPMKTPCN: GDP, Market Prices, LCU mn
Formular: FRML <IDENT> PAKNYGDPMKTPCN = PAKNECONPRVTCN+PAKNECONGOVTCN+PAKNEGDIFTOTCN+PAKNEGDISTKBCN+PAKNEEXPGNFSCN-PAKNEIMPGNFSCN+PAKNYGDPDISCCN+PAKADAP*PAKDISPREPCN $

PAKNYGDPMKTPCN: GDP, Market Prices, LCU mn
PAKADAP       : 
PAKDISPREPCN  : 
PAKNECONGOVTCN: Govt. Cons., LCU mn
PAKNECONPRVTCN: Pvt. Cons., LCU mn
PAKNEEXPGNFSCN: Exp., GNFS (NIA), millions LCU
PAKNEGDIFTOTCN: Fixed Domestic Inv., LCU mn
PAKNEGDISTKBCN: Change in stock, LCU mn
PAKNEIMPGNFSCN: Imp., GNFS (NIA), LCU mn
PAKNYGDPDISCCN: GDP Disc., LCU mn

Values :


,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
Base,"46,180,354.38","51,579,657.84","56,985,483.87","62,594,067.46","68,528,624.22","74,841,107.69","81,534,690.39","88,606,943.10","96,077,333.67","103,995,385.88","112,434,273.40","121,479,584.07","131,220,349.58","141,745,097.05","153,142,299.77","165,503,150.02","178,924,737.94","193,512,462.91","209,381,544.91","226,657,745.58","245,477,825.40","265,990,074.87","288,355,119.80","312,747,059.02","339,354,898.82","368,384,181.75","400,058,773.44","434,622,752.61","472,342,412.83","513,508,380.90","558,437,878.73"
Last,"46,180,354.38","51,579,657.84","56,985,483.88","62,594,067.47","68,528,624.22","76,456,563.74","83,849,698.36","91,343,760.41","99,049,769.25","107,102,706.85","115,610,352.28","124,693,189.28","134,461,553.55","145,015,555.20","156,450,187.67","168,861,626.86","182,351,452.41","197,028,933.46","213,012,088.36","230,428,164.33","249,414,160.67","270,117,688.27","292,698,202.00","317,328,542.35","344,196,617.57","373,507,169.29","405,483,577.74","440,369,725.97","478,431,960.58","519,961,192.42","565,275,170.74"
Diff,0.00,0.00,0.01,0.01,0.01,"1,615,456.05","2,315,007.97","2,736,817.31","2,972,435.59","3,107,320.98","3,176,078.88","3,213,605.21","3,241,203.97","3,270,458.15","3,307,887.90","3,358,476.83","3,426,714.47","3,516,470.55","3,630,543.45","3,770,418.75","3,936,335.26","4,127,613.40","4,343,082.20","4,581,483.33","4,841,718.75","5,122,987.54","5,424,804.30","5,746,973.36","6,089,547.74","6,452,811.52","6,837,292.01"


Input last run:


,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
PAKADAP,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
PAKDISPREPCN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
PAKNECONGOVTCN,"5,556,614.46","6,367,939.34","7,216,641.71","8,120,204.64","9,082,310.70","10,821,102.93","12,203,133.80","13,494,241.20","14,787,018.03","16,122,188.31","17,510,775.06","18,971,313.19","20,519,431.01","22,168,291.12","23,929,718.33","25,815,610.25","27,838,762.51","30,013,390.77","32,355,432.72","34,882,678.84","37,614,831.91","40,573,554.79","43,782,539.22","47,267,610.61","51,056,858.37","55,180,794.61","59,672,535.10","64,568,005.90","69,906,178.11","75,729,334.87","82,083,373.24"
PAKNECONPRVTCN,"39,450,086.71","43,735,888.95","47,954,011.20","52,344,927.02","57,045,147.67","63,076,447.80","68,578,723.44","74,358,087.17","80,378,603.17","86,715,179.09","93,450,257.95","100,668,700.92","108,447,209.50","116,855,310.68","125,960,538.88","135,833,807.90","146,552,702.40","158,202,774.53","170,877,728.17","184,679,236.83","199,716,980.26","216,109,123.15","233,983,205.33","253,477,331.24","274,741,480.67","297,938,883.26","323,247,423.49","350,861,101.20","380,991,585.35","413,869,898.24","449,748,254.65"
PAKNEEXPGNFSCN,"3,628,934.80","4,153,778.82","4,740,216.42","5,380,480.49","6,071,922.73","6,810,256.02","7,619,191.55","8,477,732.84","9,390,249.84","10,360,796.59","11,394,842.92","12,499,037.12","13,680,994.33","14,949,070.93","16,312,247.73","17,780,114.12","19,362,911.20","21,071,593.91","22,917,895.19","24,914,391.00","27,074,570.87","29,412,920.12","31,945,017.08","34,687,645.95","37,658,923.35","40,878,436.20","44,367,389.09","48,148,760.24","52,247,466.53","56,690,538.74","61,507,308.27"
PAKNEGDIFTOTCN,"5,357,846.53","5,945,053.34","6,516,455.63","7,079,977.54","7,649,429.94","8,756,122.51","9,535,966.13","10,288,945.01","11,064,480.23","11,887,870.26","12,773,927.57","13,742,038.29","14,811,220.75","15,999,382.57","17,323,328.20","18,799,297.35","20,443,504.96","22,272,645.35","24,304,320.03","26,557,376.61","29,052,198.26","31,810,977.31","34,857,993.43","38,219,908.46","41,926,070.36","46,008,826.78","50,503,844.80","55,450,439.91","60,891,918.63","66,875,942.29","73,454,917.56"
PAKNEGDISTKBCN,"655,457.50","711,986.84","773,391.50","840,091.95","912,544.92","991,246.53","1,076,735.69","1,169,597.79","1,270,468.69","1,380,039.12","1,499,059.34","1,628,344.36","1,768,779.44","1,921,326.22","2,087,029.27","2,267,023.22","2,462,540.58","2,674,920.17","2,905,616.24","3,156,208.48","3,428,412.82","3,724,093.19","4,045,274.23","4,394,155.25","4,773,125.19","5,184,779.06","5,631,935.64","6,117,656.83","6,645,268.61","7,218,383.78","7,840,926.76"
PAKNEIMPGNFSCN,"8,468,585.89","9,334,989.73","10,215,232.88","11,171,614.50","12,232,732.08","13,998,612.43","15,164,052.68","16,444,844.06","17,841,051.21","19,363,367.06","21,018,511.15","22,816,245.23","24,766,082.17","26,877,827.08","29,162,675.54","31,634,226.86","34,308,970.20","37,206,392.32","40,348,905.12","43,761,728.67","47,472,834.80","51,512,981.74","55,915,828.88","60,718,110.89","65,959,842.25","71,684,552.64","77,939,552.59","84,776,240.49","92,250,459.26","100,422,908.33","109,359,612.81"
PAKNYGDPDISCCN,0.26,0.28,0.30,0.33,0.36,0.39,0.42,0.46,0.50,0.54,0.59,0.64,0.69,0.75,0.82,0.89,0.96,1.05,1.14,1.23,1.34,1.46,1.58,1.72,1.87,2.03,2.20,2.39,2.60,2.82,3.07


In [11]:
mpak['PAKNYGDPMKTPCN PAKFMLBLPOLYXN PAKCCEMISCO2?KN'].rplot(samefig=1,datatype='difpctlevel')

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…